In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

with open('../data/manifestacoes.json', 'r', encoding='utf-8') as f:
    dados = json.load(f)

df = pd.DataFrame(dados)

bow_vec = CountVectorizer(strip_accents='unicode', lowercase=True)
X_bow = bow_vec.fit_transform(df['texto'])

tfidf_vec = TfidfVectorizer(strip_accents='unicode', lowercase=True)
X_tfidf = tfidf_vec.fit_transform(df['texto'])

modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
X_emb = modelo_emb.encode(df['texto'].tolist(), convert_to_numpy=True)

id_to_idx = {row['id']: idx for idx, row in df.iterrows()}

def calcular_sim(id_a, id_b):
    idx_a, idx_b = id_to_idx[id_a], id_to_idx[id_b]
    
    sim_bow = cosine_similarity(X_bow[idx_a], X_bow[idx_b])[0][0]
    sim_tfidf = cosine_similarity(X_tfidf[idx_a], X_tfidf[idx_b])[0][0]
    sim_emb = cosine_similarity(X_emb[idx_a].reshape(1, -1), X_emb[idx_b].reshape(1, -1))[0][0]
    
    return {'BoW': round(sim_bow, 4), 'TF-IDF': round(sim_tfidf, 4), 'Embeddings': round(sim_emb, 4)}

pares = [
    ('M003', 'M017', 'Mesmo tema com palavras diferentes'),
    ('M008', 'M022', 'Saúde - temas correlatos'),
    ('M008', 'M031', 'Saúde vs Iluminação - temas distantes')
]

resultados = []
for id1, id2, desc in pares:
    res = calcular_sim(id1, id2)
    res['Par'] = f"{id1} x {id2}"
    res['Descrição'] = desc
    resultados.append(res)

df_res = pd.DataFrame(resultados)[['Par', 'Descrição', 'BoW', 'TF-IDF', 'Embeddings']]
df_res

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4806.91it/s]


,Par,Descrição,BoW,TF-IDF,Embeddings
0,M003 x M017,Mesmo tema com palavras diferentes,0.0,0.0,0.2162
1,M008 x M022,Saúde - temas correlatos,0.0,0.0,0.2859
2,M008 x M031,Saúde vs Iluminação - temas distantes,0.0,0.0,0.0061
